# Uncertainty Quantification in Neural Networks
### Variational FNNs and Deep Ensembles — CHF Case Study

Standard feedforward neural networks give a single point prediction with no sense of confidence. Two practical approaches add **uncertainty estimates**:

| Method | Idea | Cost |
|---|---|---|
| **Variational FNN (vFNN)** | Replace the output layer with a Bayesian layer that samples its weights each forward pass | ~1× training cost |
| **Deep Ensemble** | Train *N* independent networks with different random seeds; spread = disagreement | *N*× training cost |

We will work through both on the **Critical Heat Flux (CHF)** dataset that comes built into pyMAISE.

**Dataset**: 2000 training + 500 test synthetic samples generated from the NEA CHF benchmark.  
**Inputs**: diameter *D*, heated length *L*, pressure *P*, mass flux *G*, inlet temperature *T$_{in}$*, outlet quality *X$_e$*  
**Output**: critical heat flux (kW m⁻²)

## 1. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score

import pyMAISE as mai
from pyMAISE.datasets import load_chf
from pyMAISE.preprocessing import scale_data

plt.rcParams.update({"font.size": 12, "figure.figsize": (7, 6)})

mai.init(
    problem_type=mai.ProblemType.REGRESSION,
    verbosity=1,
    num_configs_saved=2,
    random_state=42,
    cuda_visible_devices="-1",  # CPU only
    run_parallel=False,
)

In [ ]:
# ── Dataset toggle ────────────────────────────────────────────────────────────
# Set to True to use the real NEA CHF data (chf_train.csv / chf_valid.csv).
# Requires both files to be present in the working directory.
# Leave as False for class — uses the built-in synthetic dataset.
#
# Note: the real data has 5 inputs (no Xe / equilibrium quality column).
#       The synthetic data has 6 inputs (includes Xe).
#       The network input size is inferred automatically — no other changes needed.
USE_REAL_DATA = False

REAL_TRAIN = "chf_train.csv"
REAL_VALID  = "chf_valid.csv"

## 2. Load and Scale Data

`load_chf()` returns xarray DataArrays with named coordinates — pyMAISE works natively with these.  
We scale both inputs and outputs to [0, 1] with `MinMaxScaler` and keep the output scaler (`yscaler`) so we can invert predictions back to physical units.

In [ ]:
if USE_REAL_DATA:
    from pyMAISE.datasets import read_csv
    train_data, xtrain, ytrain = read_csv(REAL_TRAIN, input_slice=slice(0, 5), output_slice=slice(5, 6))
    test_data,  xtest,  ytest  = read_csv(REAL_VALID,  input_slice=slice(0, 5), output_slice=slice(5, 6))
else:
    train_data, xtrain, ytrain, test_data, xtest, ytest = load_chf()

print("Input features:", list(xtrain.coords["variable"].values))
print("Output:        ", list(ytrain.coords["variable"].values))
print(f"Train shape: {xtrain.shape}   Test shape: {xtest.shape}")

In [ ]:
xtrain, xtest, _ = scale_data(xtrain, xtest, MinMaxScaler())
ytrain, ytest, yscaler = scale_data(ytrain, ytest, MinMaxScaler())

split_data = (xtrain, xtest, ytrain, ytest)

## 3. Variational FNN

### 3.1 How it works

A standard dense layer learns a fixed weight matrix **W**.  
A variational (Bayesian) layer instead learns a **distribution** over weights:

$$q(\mathbf{w}) = \mathcal{N}(\boldsymbol{\mu}_w,\; \text{softplus}(\boldsymbol{\rho}_w))$$

At each forward pass a fresh weight sample is drawn, so running the same input through the network 100 times gives 100 slightly different predictions. The spread of those predictions is an approximation of the **epistemic (model) uncertainty**.

The training loss adds a KL divergence penalty that keeps the learned posterior close to the prior:

$$\mathcal{L} = \underbrace{\text{MSE}}_{\text{fit}} + \beta\, \underbrace{D_{\text{KL}}(q \| p)}_{\text{regularisation}}$$

Setting $\beta = 1/N_{\text{train}}$ puts both terms on the same per-sample scale.

### 3.2 Architecture in pyMAISE

Use `Variational_<name>` as the layer key — pyMAISE routes it to `DenseReparameterization` from TensorFlow Probability. All other layers are standard Keras `Dense`.

Our architecture: two deterministic hidden layers → one variational output layer.

**Hyperparameter to tune**: `posterior_scale_init` controls how wide the weight posterior starts.  
The effective initial standard deviation is $\sigma = \text{softplus}(\rho_0)$:

| `posterior_scale_init` | $\sigma$ | Effect |
|---|---|---|
| −3.0 (TFP default) | ≈ 0.049 | More uncertainty from the start |
| −4.0 | ≈ 0.018 | Starts near-deterministic; often converges better |

We let pyMAISE grid-search both with `mai.Choice`.

In [ ]:
kl_weight = 1.0 / xtrain.shape[0]  # 1/N_train — correct ELBO scaling

model_settings = {
    "models": ["vfnn"],
    "vfnn": {
        "structural_params": {
            "Dense_hidden1": {"units": 64, "activation": "tanh"},
            "Dense_hidden2": {"units": 64, "activation": "tanh"},
            "Variational_output": {
                "units": 1,
                "activation": "linear",
                "kl_weight": kl_weight,
                "posterior_scale_init": mai.Choice([-3.0, -4.0]),
            },
        },
        "optimizer": "Adam",
        "Adam": {"learning_rate": 0.001},
        "compile_params": {"loss": "mean_squared_error", "metrics": ["mean_squared_error"]},
        "fitting_params": {
            "epochs": 60,
            "batch_size": 64,
            "validation_split": 0.15,
            "kl_schedule": "sigmoid_growth",  # ramp KL penalty from 0→1 over training
        },
    },
}

### 3.3 Grid Search

Grid search runs one trial per `posterior_scale_init` value and keeps the top 2.

In [ ]:
tuner = mai.Tuner(xtrain, ytrain, model_settings=model_settings)
configs = tuner.nn_grid_search(objective="r2_score", cv=2)

### 3.4 PostProcessor — built-in tools

Pass `n_mc_samples=200` to run 200 stochastic forward passes for every test point.  
pyMAISE stores the full sample array and can use it for uncertainty plots.

In [ ]:
pp = mai.PostProcessor(
    data=split_data,
    model_configs=[configs],
    yscaler=yscaler,
    n_mc_samples=200,
)

In [ ]:
# Summary metrics table (R², MAE, RMSE, …)
pp.metrics().drop("Parameter Configurations", axis=1)

In [ ]:
# Built-in variational parity plot — 90% and 95% CI bands from MC samples
fig, ax = plt.subplots(figsize=(7, 7))
pp.variational_parity_plot(ax=ax, model_type="vfnn")
ax.set_title("vFNN — parity plot")
ax.set_xlabel("True CHF (kW m⁻²)")
ax.set_ylabel("Predicted CHF (kW m⁻²)")
plt.tight_layout()
plt.show()

In [ ]:
# Learning curve
pp.nn_learning_plot(model_type="vfnn")
plt.show()

## 4. v2FNN — Two Variational Layers

The vFNN above only places uncertainty on the **last** layer. The deterministic hidden layers map inputs to a fixed representation, so different inputs that land in the same region of hidden-layer space will get similar uncertainty estimates regardless of how different they are.

**v2FNN** pushes Bayesian layers deeper: one deterministic hidden layer extracts basic features, then a variational hidden layer and a variational output layer both sample weights at inference time. This gives the uncertainty estimate two chances to respond to the input.

Architecture:

```
Input (6)
  → Dense_hidden1   (64, tanh)          — deterministic feature extraction
  → Variational_hidden2  (64, tanh)     — Bayesian layer #1
  → Variational_output   (1, linear)    — Bayesian layer #2
```

Because two variational layers each contribute a KL term, we halve `kl_weight` per layer so the total KL penalty stays on the same scale as the vFNN above.

In [ ]:
kl_weight_v2 = 0.5 / xtrain.shape[0]  # halved — two variational layers share the KL budget

v2fnn_settings = {
    "models": ["v2fnn"],
    "v2fnn": {
        "structural_params": {
            "Dense_hidden1": {"units": 64, "activation": "tanh"},
            "Variational_hidden2": {
                "units": 64,
                "activation": "tanh",
                "kl_weight": kl_weight_v2,
                "posterior_scale_init": -4.0,
            },
            "Variational_output": {
                "units": 1,
                "activation": "linear",
                "kl_weight": kl_weight_v2,
                "posterior_scale_init": -4.0,
            },
        },
        "optimizer": "Adam",
        "Adam": {"learning_rate": 0.001},
        "compile_params": {"loss": "mean_squared_error", "metrics": ["mean_squared_error"]},
        "fitting_params": {
            "epochs": 60,
            "batch_size": 64,
            "validation_split": 0.15,
            "kl_schedule": "sigmoid_growth",
        },
    },
}

In [ ]:
tuner_v2 = mai.Tuner(xtrain, ytrain, model_settings=v2fnn_settings)
configs_v2 = tuner_v2.nn_grid_search(objective="r2_score", cv=2)

In [ ]:
pp_v2 = mai.PostProcessor(
    data=split_data,
    model_configs=[configs_v2],
    yscaler=yscaler,
    n_mc_samples=200,
)

In [ ]:
pp_v2.metrics().drop("Parameter Configurations", axis=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

pp.variational_parity_plot(ax=axes[0], model_type="vfnn")
axes[0].set_title("vFNN (last layer Bayesian)")
axes[0].set_xlabel("True CHF (kW m⁻²)")
axes[0].set_ylabel("Predicted CHF (kW m⁻²)")

pp_v2.variational_parity_plot(ax=axes[1], model_type="v2fnn")
axes[1].set_title("v2FNN (two Bayesian layers)")
axes[1].set_xlabel("True CHF (kW m⁻²)")
axes[1].set_ylabel("")

plt.suptitle("Parity plots: vFNN vs v2FNN", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Deep Ensemble — Preview

The simplest and often best-calibrated approach: train *N* **standard** FNNs independently with different random seeds. The spread across their predictions gives **input-dependent** uncertainty that reflects genuine disagreement between models.

- No probabilistic machinery needed — plain Keras `Dense` layers throughout
- Trivially parallelisable
- Empirically the best-calibrated approach on tabular data

We train 10 members here to keep runtime manageable in class.

In [ ]:
N_MEMBERS = 10

# Inverse-transform test labels back to physical units for R² and plotting
y_true = yscaler.inverse_transform(ytest.values.reshape(-1, 1)).flatten()

fnn_settings = {
    "models": ["fnn"],
    "fnn": {
        "structural_params": {
            "Dense_hidden1": {"units": 64, "activation": "tanh"},
            "Dense_hidden2": {"units": 64, "activation": "tanh"},
            "Dense_output":  {"units": 1,  "activation": "linear"},
        },
        "optimizer": "Adam",
        "Adam": {"learning_rate": 0.001},
        "compile_params": {"loss": "mse", "metrics": ["mean_squared_error"]},
        "fitting_params": {"epochs": 60, "batch_size": 64, "validation_split": 0.15},
    },
}

In [ ]:
member_preds = []

for seed in range(N_MEMBERS):
    mai.init(
        problem_type=mai.ProblemType.REGRESSION,
        verbosity=0,
        num_configs_saved=1,
        random_state=seed,
        cuda_visible_devices="-1",
        run_parallel=False,
    )
    t = mai.Tuner(xtrain, ytrain, model_settings=fnn_settings)
    cfg = t.nn_grid_search(objective="mean_squared_error", cv=2)
    ep = mai.PostProcessor(
        data=split_data, model_configs=[cfg], yscaler=yscaler, n_mc_samples=0
    )
    yhat = ep._models["Test Yhat"][0].flatten()
    r2 = r2_score(y_true, yhat)
    print(f"  Member {seed}  Test R²={r2:.4f}")
    member_preds.append(yhat)

In [ ]:
preds    = np.stack(member_preds, axis=0)          # (N_MEMBERS, n_test)
ens_mean = preds.mean(axis=0)
ens_lo   = np.percentile(preds, 2.5,  axis=0)
ens_hi   = np.percentile(preds, 97.5, axis=0)
ens_width = ens_hi - ens_lo

print(f"\nEnsemble Test R² = {r2_score(y_true, ens_mean):.4f}")
print(f"Median 95% PI width: {np.median(ens_width):.1f} kW m⁻²")

In [ ]:
# Parity plot with ensemble 95% PI
fig, ax = plt.subplots(figsize=(7, 7))
yerr = np.array([ens_mean - ens_lo, ens_hi - ens_mean])
ax.errorbar(y_true, ens_mean, yerr=yerr,
            fmt="o", alpha=0.4, markersize=3, elinewidth=0.7, capsize=0,
            label=f"Mean + 95% PI ({N_MEMBERS} members)")
lo, hi = min(y_true.min(), ens_lo.min()), max(y_true.max(), ens_hi.max())
ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, label="Perfect")
ax.set_xlabel("True CHF (kW m⁻²)")
ax.set_ylabel("Predicted CHF (kW m⁻²)")
ax.set_title(f"Deep Ensemble parity plot (N={N_MEMBERS})\nTest R²={r2_score(y_true, ens_mean):.4f}")
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

## 6. Ideas to explore further
- Increase ensemble size (10–20 members) and observe calibration improvement
- Have each member of the ensemble get independently tuned so that their network architectures are different
- Or randomize the network structure
- Test on other datasets